# EDA 2: Hasil Preprocessing

Notebook ini mengeksplorasi dataset hasil preprocessing dari `02_preprocessing.ipynb`. Fokus analisis adalah kualitas data, distribusi label toxicity, kategori anotasi, karakteristik panjang komentar, topik, dan emoji.

Sumber: `data/processed/data_cleaned_before_encode.csv`. Jalankan notebook preprocessing terlebih dahulu jika file belum tersedia atau ingin memperbaruinya.

In [ ]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "data_cleaned_before_encode.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset preprocessing tidak ditemukan: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
required_columns = {"text", "text_clean", "toxicity", "topic"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {sorted(missing_columns)}")

print(f"Dataset: {DATA_PATH}")
print(f"Ukuran: {df.shape[0]:,} baris x {df.shape[1]} kolom")
display(df.head())

## Kualitas Data

Periksa kelengkapan kolom utama, duplikasi teks, dan baris tanpa teks bersih sebelum membaca pola distribusi.

In [ ]:
quality_summary = pd.DataFrame(
    {
        "Metrik": [
            "Jumlah baris",
            "Teks kosong",
            "Teks bersih kosong",
            "Teks bersih duplikat",
            "Label toxicity kosong",
        ],
        "Nilai": [
            len(df),
            int(df["text"].fillna("").astype(str).str.strip().eq("").sum()),
            int(df["text_clean"].fillna("").astype(str).str.strip().eq("").sum()),
            int(df["text_clean"].duplicated().sum()),
            int(df["toxicity"].isna().sum()),
        ],
    }
)
display(quality_summary)
display(df.isna().sum().sort_values(ascending=False).rename("missing_values").to_frame().head(15))

## Distribusi Label dan Kategori

Distribusi ini membantu melihat ketidakseimbangan target serta kategori anotasi yang paling sering muncul.

In [ ]:
df["toxicity"] = pd.to_numeric(df["toxicity"], errors="coerce")
toxicity_counts = df["toxicity"].value_counts(dropna=False).sort_index()
display(toxicity_counts.rename_axis("toxicity").to_frame("jumlah"))

category_columns = [
    "profanity_obscenity",
    "threat_incitement_to_violence",
    "insults",
    "identity_attack",
    "sexually_explicit",
]
category_columns = [column for column in category_columns if column in df.columns]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
toxicity_counts.plot(kind="bar", ax=axes[0], color="#2A9D8F")
axes[0].set(title="Distribusi label toxicity", xlabel="Label", ylabel="Jumlah komentar")
axes[0].tick_params(axis="x", rotation=0)

if category_columns:
    category_rates = (
        df[category_columns]
        .apply(lambda column: pd.to_numeric(column, errors="coerce").fillna(0).mean() * 100)
        .sort_values()
    )
    category_rates.plot(kind="barh", ax=axes[1], color="#E76F51")
    axes[1].set(xlabel="Persentase komentar (%)", ylabel="Kategori", title="Prevalensi kategori anotasi")
else:
    axes[1].text(0.5, 0.5, "Kolom kategori tidak tersedia", ha="center", va="center")
    axes[1].set_axis_off()
plt.tight_layout()
plt.show()

## Panjang Komentar

Bandingkan panjang teks asli dan teks bersih. Ringkasan per label dapat memberi indikasi apakah panjang komentar berbeda pada kelas toxicity.

In [ ]:
df["raw_char_count"] = df["text"].fillna("").astype(str).str.len()
df["clean_token_count"] = df["text_clean"].fillna("").astype(str).str.split().str.len()
length_by_label = df.groupby("toxicity", dropna=False).agg(
    jumlah=("text_clean", "size"),
    median_karakter_asli=("raw_char_count", "median"),
    median_token_bersih=("clean_token_count", "median"),
).round(1)
display(length_by_label)

plt.figure(figsize=(9, 4))
sns.histplot(data=df, x="clean_token_count", hue="toxicity", bins=35, element="step", stat="density", common_norm=False)
plt.title("Distribusi jumlah token pada teks bersih")
plt.xlabel("Jumlah token")
plt.ylabel("Kepadatan")
plt.tight_layout()
plt.show()

## Eksplorasi Topik

Topik dapat tersimpan sebagai representasi list dalam CSV. Parsing dilakukan secara aman menggunakan `ast.literal_eval`, lalu komentar multi-topik diekspansi untuk menghitung frekuensi dan tingkat toxicity per topik.

In [ ]:
def parse_topic_list(value):
    if isinstance(value, list):
        return [str(topic).strip() for topic in value if str(topic).strip()]
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, list):
            return [str(topic).strip() for topic in parsed if str(topic).strip()]
    except (ValueError, SyntaxError):
        pass
    return [topic.strip() for topic in str(value).split(",") if topic.strip()]

topic_source = df["topic_list"] if "topic_list" in df.columns else df["topic"]
df["topics_eda"] = topic_source.apply(parse_topic_list)
topic_long = df[["toxicity", "topics_eda"]].explode("topics_eda")
topic_long = topic_long[topic_long["topics_eda"].notna() & topic_long["topics_eda"].ne("")]

topic_summary = (
    topic_long.groupby("topics_eda").agg(
        jumlah_komentar=("toxicity", "size"),
        proporsi_toxic=("toxicity", "mean"),
    )
    .sort_values("jumlah_komentar", ascending=False)
)
display(topic_summary.head(20).style.format({"proporsi_toxic": "{:.1%}"}))

topic_summary.head(15)["jumlah_komentar"].sort_values().plot(kind="barh", figsize=(9, 5), color="#457B9D")
plt.title("15 topik dengan komentar terbanyak")
plt.xlabel("Jumlah komentar")
plt.ylabel("Topik")
plt.tight_layout()
plt.show()

## Emoji

Tinjau berapa banyak komentar yang mengandung emoji serta contoh emoji yang paling sering diekstrak pada preprocessing.

In [ ]:
if "emoji" in df.columns:
    emoji_values = df["emoji"].fillna("").astype(str).str.split()
    emoji_counts = emoji_values.explode().dropna()
    emoji_counts = emoji_counts[emoji_counts.ne("")].value_counts().head(20)
    emoji_summary = pd.Series(
        {
            "Komentar dengan emoji": int(df["emoji"].fillna("").astype(str).str.strip().ne("").sum()),
            "Total komentar": len(df),
        }
    )
    display(emoji_summary.to_frame("jumlah"))
    if not emoji_counts.empty:
        display(emoji_counts.rename("jumlah").to_frame())
        emoji_counts.sort_values().plot(kind="barh", figsize=(8, 5), color="#F4A261")
        plt.title("20 emoji paling sering")
        plt.xlabel("Jumlah kemunculan")
        plt.ylabel("Emoji")
        plt.tight_layout()
        plt.show()
else:
    print("Kolom emoji tidak tersedia pada dataset preprocessing.")

## Catatan Analisis

Gunakan tabel dan grafik di atas untuk mencatat temuan utama sebelum pemodelan. Persentase toxicity per topik bersifat deskriptif dan tidak menunjukkan hubungan sebab-akibat. Pastikan distribusi label serta ukuran kelas diperhitungkan saat menentukan strategi split dan evaluasi model.